# **CS6886W: Systems Engineering for Deep Learning - Assignment 2**
### **MobileNet-v2 Compression & Custom Quantization on CIFAR-10**
**Department of Computer Science and Engineering, IIT Madras**  
**Student Name**: Prasanna JS  
**Roll Number**: CS25M524  

---
### **Workflow Overview**
1. **Mount Google Drive** to persist checkpoints and artifacts across sessions.
2. **Train Baseline MobileNet-v2** on CIFAR-10 (stride=1 adaptation, Cosine LR scheduler, Data Augmentation + Cutout).
3. **Evaluate In-Memory Compression**: Custom uniform weight & activation fake-quantization without external compression APIs.
4. **Run Quantization Sweeps (≥ 8 Runs)** logged to **Weights & Biases (W&B)** to generate the Parallel Coordinates chart.
5. **Analyze Activation & Weight Memory Footprints** using PyTorch forward hooks.

## **Step 1: Check GPU & Mount Google Drive**

In [ ]:
# Check GPU accelerator
!nvidia-smi

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## **Step 2: Clone/Navigate to Repository & Install Dependencies**

In [ ]:
# Set project directory (Adjust path to where you placed the repo in Drive or local Colab storage)
import os
PROJECT_DIR = '/content/drive/MyDrive/CS6886W_Assignment2/mobilenetv2_compression'

if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}
else:
    print("Path not found in Drive. Running from current working directory.")

# Install dependencies
!pip install wandb matplotlib -q

## **Step 3: Weights & Biases (W&B) Setup**

In [ ]:
import wandb
# Paste your W&B API Key here or login interactively
# os.environ["WANDB_API_KEY"] = "YOUR_KEY_HERE"
wandb.login()

## **Step 4: Train Baseline MobileNet-v2 (Question 1)**
Trains MobileNetV2 with `nesterov_sgd`, `lr=0.05`, `batch_size=128`, for 100 epochs with Cosine Annealing scheduler.
Saves the best checkpoint to `./checkpoints/best_mobilenetv2.pth` and plots loss/accuracy curves to `./data/training_curves.png`.

In [ ]:
!python main.py \
    --epochs 100 \
    --batch_size 128 \
    --lr 0.05 \
    --optimizer nesterov_sgd \
    --width_mult 1.0 \
    --dropout 0.2 \
    --seed 42

### **Display Training & Validation Curves (Question 1c)**

In [ ]:
from IPython.display import Image, display
if os.path.exists('./data/training_curves.png'):
    display(Image('./data/training_curves.png', width=800))
else:
    print("Training curves not found. Make sure main.py has completed.")

## **Step 5: Evaluate Baseline & W8A8 Quantization (Question 2 & 4)**
Runs custom uniform quantization on weights and activations, computes metadata overheads, and calculates in-memory RAM/VRAM footprints.

In [ ]:
!python test.py \
    --weight_quant_bits 8 \
    --activation_quant_bits 8 \
    --checkpoint ./checkpoints/best_mobilenetv2.pth \
    --calib_batches 20

## **Step 6: Run Comprehensive Quantization Sweep (Question 3)**
Executes 12 simulation configurations across varying weight bit-widths (8, 6, 4), activation bit-widths (8, 6, 4), and per-channel vs. per-tensor scaling.
Logs all metrics into W&B to create the **Parallel Coordinates Chart**.

In [ ]:
!python sweep_wandb.py \
    --checkpoint ./checkpoints/best_mobilenetv2.pth \
    --calib_batches 20

## **Step 7: Deep Activation Memory Footprint Inspection (Question 4b)**
Demonstrates the exact PyTorch forward hooks mechanism used to measure the layer-by-layer activation footprint during an inference forward pass.

In [ ]:
import torch
from model import get_model
from quantize import swap_to_quant_modules, freeze_all_quant, compute_model_in_memory_size
from utils import measure_activation_memory

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_model(num_classes=10)
model.to(device)

# Check activation memory for 8-bit, 6-bit, and 4-bit activations
for a_bits in [8, 6, 4]:
    res = measure_activation_memory(model, input_shape=(1, 3, 32, 32), act_bits=a_bits, device=device)
    print(f"[Batch Size 1] Act Bits: {a_bits} | FP32 Act RAM: {res['fp32_kb']:.2f} KB | Quant Act RAM: {res['quant_kb']:.2f} KB | Comp Ratio: {res['compression_ratio']:.2f}x")

# Layer breakdown sample
print("\n--- First 10 Layer Activation Shapes ---")
for layer_name, num_elements in res['layer_breakdown'][:10]:
    print(f"{layer_name:25s}: {num_elements:,} elements")